In [0]:
%sql
MERGE INTO project1catalog.edw_gold.dim_category AS target
USING (
  SELECT DISTINCT
    category AS Category,
    subcategory AS Subcategory
  FROM project1catalog.edw_silver.servicenow_event_log
) AS source
ON target.Category = source.Category
   AND target.Subcategory = source.Subcategory
WHEN NOT MATCHED THEN
  INSERT (Category, Subcategory)
  VALUES (source.Category, source.Subcategory);

In [0]:
%sql
MERGE INTO project1catalog.edw_gold.dim_user AS target
USING (
  SELECT DISTINCT
    opened_by  AS opened_by
  FROM project1catalog.edw_silver.servicenow_event_log
  WHERE opened_by IS NOT NULL
) AS source
ON target.openedby = source.opened_by
WHEN NOT MATCHED AND source.opened_by IS NOT NULL THEN
  INSERT (openedby, SysCreatedBy, SysUpdatedBy)
  VALUES (source.opened_by, NULL, NULL);

In [0]:
%sql
MERGE INTO project1catalog.edw_gold.fact_event_log AS target
USING (
  SELECT *
  FROM (
    SELECT
      s.number AS Number,
      u.UserId,
      c.CategoryId,
      to_timestamp(s.opened_at, 'd/M/yyyy H:mm') AS OpenedAt,
      ROW_NUMBER() OVER (
        PARTITION BY s.number, u.UserId, c.CategoryId
        ORDER BY s.opened_at DESC
      ) AS rn
    FROM project1catalog.edw_silver.servicenow_event_log s
    LEFT JOIN project1catalog.edw_gold.dim_user u
      ON s.opened_by = u.OpenedBy
    LEFT JOIN project1catalog.edw_gold.dim_category c
      ON s.category = c.Category
      AND s.subcategory = c.Subcategory
  ) deduped
  WHERE rn = 1
) AS source
ON target.Number = source.Number
   AND target.UserId = source.UserId
   AND target.CategoryId = source.CategoryId
WHEN MATCHED THEN
  UPDATE SET
    target.OpenedAt = source.OpenedAt
WHEN NOT MATCHED THEN
  INSERT (Number, UserId, CategoryId, OpenedAt)
  VALUES (source.Number, source.UserId, source.CategoryId, source.OpenedAt);